In [ ]:
from pyspark.sql.functions import (
    col, sum as _sum, avg as _avg, count, when, round as _round,
    current_timestamp
)

In [ ]:
# === Gold: Grid Health (hourly per substation) ===
silver_sensors = spark.read.format('delta').table('silver_grid_sensors')

gold_grid_health = (
    silver_sensors
    .withColumnRenamed('sensor_date', 'date')
    .withColumnRenamed('sensor_hour', 'hour')
    .groupBy('date', 'hour', 'substation_id', 'region')
    .agg(
        _round(_avg('voltage_v'), 2).alias('avg_voltage'),
        _round(_avg('frequency_hz'), 3).alias('avg_frequency'),
        _round(_avg('load_mw'), 2).alias('avg_load'),
        _round(_avg('power_factor'), 3).alias('avg_power_factor'),
        _round(_avg('temperature_c'), 1).alias('avg_temperature'),
        count('*').alias('reading_count'),
        _sum(when(col('voltage_anomaly'), 1).otherwise(0)).alias('voltage_anomalies'),
    )
    .withColumn('gold_timestamp', current_timestamp())
)

gold_grid_health.write.mode('overwrite').format('delta').saveAsTable('gold_grid_health')
print(f'Gold grid health: {spark.table("gold_grid_health").count()} rows')

In [ ]:
# === Gold: Outage Summary (daily per region) ===
silver_events = spark.read.format('delta').table('silver_power_events')

gold_outage_summary = (
    silver_events
    .withColumnRenamed('event_date', 'date')
    .groupBy('date', 'region')
    .agg(
        count('*').alias('total_events'),
        _sum(when(col('event_type') == 'outage', 1).otherwise(0)).alias('outages'),
        _sum(when(col('event_type') == 'surge', 1).otherwise(0)).alias('surges'),
        _sum(when(col('event_type') == 'voltage_sag', 1).otherwise(0)).alias('sags'),
        _sum(when(col('resolved').cast('boolean') == False, 1).otherwise(0)).alias('faults'),
        _sum('affected_customers').alias('total_affected'),
        _round(_avg(col('duration_sec') / 60.0), 1).alias('avg_duration_min'),
        _sum(when(col('severity') == 'critical', 1).otherwise(0)).alias('critical_events'),
    )
    .withColumn('gold_timestamp', current_timestamp())
)

gold_outage_summary.write.mode('overwrite').format('delta').saveAsTable('gold_outage_summary')
print(f'Gold outage summary: {spark.table("gold_outage_summary").count()} rows')

In [ ]:
# === Gold: Renewable Summary (daily per plant type) ===
silver_renewable = spark.read.format('delta').table('silver_renewable_generation')

gold_renewable_summary = (
    silver_renewable
    .withColumnRenamed('gen_date', 'date')
    .groupBy('date', 'plant_type')
    .agg(
        _round(_sum('generation_mw'), 2).alias('total_generation_mw'),
        _round(_avg('capacity_factor'), 3).alias('avg_capacity_factor'),
        _round(_sum('capacity_mw'), 2).alias('total_capacity_mw'),
        count('*').alias('readings'),
    )
    .withColumn('gold_timestamp', current_timestamp())
)

gold_renewable_summary.write.mode('overwrite').format('delta').saveAsTable('gold_renewable_summary')
print(f'Gold renewable summary: {spark.table("gold_renewable_summary").count()} rows')

In [ ]:
# Optimize all Gold tables
for table in ['gold_grid_health', 'gold_outage_summary', 'gold_renewable_summary']:
    spark.sql(f'OPTIMIZE {table}')
    cnt = spark.read.format('delta').table(table).count()
    print(f'{table}: {cnt} rows (optimized)')

print('\n=== Gold Layer Complete — 3 tables ===')